<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<ejemplo de redes bayesianas>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  
# Simulación para determinar si un estudiante aprueba o desaprueba una asignatura.
# La red relaciona las variables de la siguiente manera
HorasEstudio → Asistencia → NotaExamen → Aprobacion

In [1]:
#!pip install pgmpy

In [2]:
# libreria de redes bayesianas
# crear la red bayesiana
from pgmpy.models import DiscreteBayesianNetwork
# crear una tabla de probabilidades condicionales
from pgmpy.factors.discrete import TabularCPD
# realizar inferencia en la red bayesiana
from pgmpy.inference import VariableElimination

c:\Users\Haand\AppData\Local\Programs\Python\Python314\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [3]:
# crear la red bayesiana
model = DiscreteBayesianNetwork([
    ("HorasEstudio", "Asistencia"),
    ("Asistencia", "NotaExamen"),
    ("NotaExamen", "Aprobacion")
])

In [4]:
# crear la tabla de probabilidades condicionales para la variable HorasEstudio

# que puede 60% de probabilidad de que el estudiante estudie pocas horas
# y 40% de probabilidad de que el estudiante estudie muchas horas

cpd_horas_estudio = TabularCPD(
    variable="HorasEstudio",
    variable_card=2,
    values=[[0.60], [0.40]],
    state_names={"HorasEstudio": ["Pocas", "Muchas"]}
)

In [5]:
# crear la tabla de probabilidades para la variable Asistencia
# es una hipotesis de incertidumbre,
# que puede 80% de probabilidad de que la asistencia sea "Baja" si estudia pocas horas
# y 20% de probabilidad de que la asistencia sea "Alta" si estudia pocas horas
# que puede 30% de probabilidad de que la asistencia sea "Baja" si estudia muchas horas
# y 70% de probabilidad de que la asistencia sea "Alta" si estudia muchas horas

cpd_asistencia = TabularCPD(
    variable="Asistencia",
    variable_card=2,
    values=[[0.80, 0.30], [0.20, 0.70]],
    evidence=["HorasEstudio"],
    evidence_card=[2],
    state_names={
        "Asistencia": ["Baja", "Alta"],
        "HorasEstudio": ["Pocas", "Muchas"]
    }
)

In [6]:
# crear la tabla de probabilidades condicionales para la variable NotaExamen
# que puede 70% de probabilidad de que la nota del examen sea "Baja" si la asistencia es baja
# y 30% de probabilidad de que la nota sea "Alta" si la asistencia es baja
# que puede 20% de probabilidad de que la nota del examen sea "Baja" si la asistencia es alta
# y 80% de probabilidad de que la nota sea "Alta" si la asistencia es alta

cpd_nota_examen = TabularCPD(
    variable="NotaExamen",
    variable_card=2,
    values=[[0.70, 0.20], [0.30, 0.80]],
    evidence=["Asistencia"],
    evidence_card=[2],
    state_names={
        "NotaExamen": ["Baja", "Alta"],
        "Asistencia": ["Baja", "Alta"]
    }
)

In [7]:
# crear la tabla de probabilidades condicionales para la variable Aprobacion
# que puede 95% de probabilidad de que el estudiante desapruebe si la nota del examen es baja
# y 5% de probabilidad de que apruebe si la nota del examen es baja
# que puede 10% de probabilidad de que el estudiante desapruebe si la nota del examen es alta
# y 90% de probabilidad de que apruebe si la nota del examen es alta

cpd_aprobacion = TabularCPD(
    variable="Aprobacion",
    variable_card=2,
    values=[[0.95, 0.10], [0.05, 0.90]],
    evidence=["NotaExamen"],
    evidence_card=[2],
    state_names={
        "Aprobacion": ["Desaprueba", "Aprueba"],
        "NotaExamen": ["Baja", "Alta"]
    }
)

In [8]:
# agregar al modelo las tablas de probabilidades condicionales
model.add_cpds(cpd_horas_estudio, cpd_asistencia, cpd_nota_examen, cpd_aprobacion)
model.check_model()

True

In [9]:
print("Modelo de red bayesiana creado y verificado correctamente.")
print(model.edges())

Modelo de red bayesiana creado y verificado correctamente.
[('HorasEstudio', 'Asistencia'), ('Asistencia', 'NotaExamen'), ('NotaExamen', 'Aprobacion')]


In [10]:
# Inferencia en la red bayesiana
# motor o proceso de inferencia para el grafo
inferencia = VariableElimination(model)

resultado_aprobacion = inferencia.query(
    variables=["Aprobacion"],
    evidence={"HorasEstudio": "Muchas", "Asistencia": "Alta"}
)

print("Probabilidad de que el estudiante apruebe dado que estudia muchas horas y tiene asistencia alta:")
print(resultado_aprobacion)

Probabilidad de que el estudiante apruebe dado que estudia muchas horas y tiene asistencia alta:
+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Desaprueba) |            0.2700 |
+------------------------+-------------------+
| Aprobacion(Aprueba)    |            0.7300 |
+------------------------+-------------------+


## si el estudiante estudia muchas horas y tiene asistencia alta, la red bayesiana estima un 73% de probabilidad de que apruebe y un 27% de que desapruebe.